# 01 — Data Exploration: Cryptocurrency Log Returns

First look at the Phase 1 dataset: full-history daily OHLCV for BTC, ETH, SOL, DOGE
and USDC (stablecoin contrast) from the Binance public data dump.

**What we're looking for** (this motivates the Phase 2 model choices):
- *Non-stationary prices vs. (roughly) stationary log returns* → why we model returns, and why ARIMA needs the "I".
- *Volatility clustering* → motivation for GARCH.
- *Cross-asset co-movement* → motivation for VAR.

In [ ]:
import sys; sys.path.insert(0, "..")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.dataloader import load_all, COINS

daily = load_all("1d")
for sym, df in daily.items():
    print(f"{sym:9s} {COINS[sym]:28s} {len(df):5d} days   "
          f"{df.timestamp.min().date()} → {df.timestamp.max().date()}")

## Prices — levels are clearly non-stationary

In [ ]:
fig, axes = plt.subplots(len(daily), 1, figsize=(11, 2.4 * len(daily)))
for ax, (sym, df) in zip(axes, daily.items()):
    ax.plot(df.timestamp, df.close, lw=0.8)
    ax.set_title(f"{sym} close (daily)"); ax.set_yscale("log"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Log returns — roughly mean-zero, but volatility clusters in time

Quiet periods alternate with bursts of turbulence. That persistence in *squared* returns
is exactly what GARCH is built to capture.

In [ ]:
fig, axes = plt.subplots(len(daily), 1, figsize=(11, 2.4 * len(daily)))
for ax, (sym, df) in zip(axes, daily.items()):
    ax.plot(df.timestamp, df.log_return, lw=0.5)
    ax.set_title(f"{sym} daily log returns"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(len(daily), 1, figsize=(11, 2.4 * len(daily)))
for ax, (sym, df) in zip(axes, daily.items()):
    rv = df.log_return.rolling(30).std() * np.sqrt(365)  # annualized 30d vol
    ax.plot(df.timestamp, rv, lw=0.9, color="darkred")
    ax.set_title(f"{sym} 30-day rolling annualized volatility"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Summary statistics — fat tails everywhere

Excess kurtosis ≫ 0 confirms heavy tails (a Gaussian ARMA understates crash risk;
this is where GARCH-type conditional volatility helps). Note DOGE (kurtosis ≈ 143) and
USDC (≈ 218): the stablecoin is near-zero volatility punctuated by rare depeg events.

In [ ]:
rows = []
for sym, df in daily.items():
    r = df.log_return.dropna()
    rows.append({"symbol": sym, "obs": len(r), "mean": r.mean(), "std": r.std(),
                 "skew": r.skew(), "kurtosis": r.kurtosis(), "min": r.min(), "max": r.max()})
pd.DataFrame(rows).set_index("symbol").round(4)

## Cross-asset correlation — why VAR is interesting here

The four non-stable coins are strongly positively correlated (BTC–ETH ≈ 0.81); USDC is
essentially uncorrelated. A VAR on BTC/ETH/SOL returns models how a shock to one
propagates to the others (impulse response functions in Phase 2/4).

In [ ]:
rets = pd.DataFrame({sym: df.set_index("timestamp").log_return
                     for sym, df in daily.items()}).dropna()
rets.corr().round(3)

### Takeaways for Phase 2

1. Model **log returns**, not price levels (non-stationarity).
2. Returns show **volatility clustering + fat tails** → GARCH family.
3. Strong **cross-asset correlation** → VAR for joint dynamics + impulse responses.
4. USDC serves as a sanity-check contrast asset.